# Weekly static snapshots — Eco-Counter & Vivacity

Run this notebook **locally once a month** to refresh two JSON files under `static/data/`.

## Outputs (one file per source)

| File | Contents |
|------|----------|
| `static/data/eco-counter-weekly-snapshot.json` | Per-site weekly totals (pedestrian + bike) |
| `static/data/vivacity-weekly-snapshot.json` | Per-sensor weekly totals (pedestrian + bike) |

**No daily or monthly rows are stored.** Sum weekly rows to get monthly or network totals downstream.

History window: from the **start of the same calendar month one year before the last completed month** (e.g. run in July 2026 → fetch from June 2025) so YoY KPIs can compare June 2026 vs June 2025.

## Setup

1. From repo root: `pip install requests python-dotenv`
2. Set in environment or `.env`: `ECO_COUNTER_API`, `VIVACITY_API`
3. Run all cells from repo root or `notebooks/`

In [2]:
# %pip install -q requests python-dotenv


In [3]:
import json
import os
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from pathlib import Path
from urllib.parse import urlencode

import requests

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# ── Configuration ─────────────────────────────────────────────────────────────
# History start is computed in helpers: last completed month, same month last year.
# Eco-Counter rate limits aggressively — keep concurrency low and pause between calls.
ECO_CONCURRENCY = 1
ECO_PAUSE_S = 1.5
ECO_429_MAX_RETRIES = 6
ECO_429_BACKOFF_S = 30     # doubles each retry: 30s, 60s, 120s, …

VIVACITY_WINDOW_MAX_DAYS = 60
VIVACITY_PAUSE_S = 0.8
VIVACITY_TIMEOUT_S = 300
VIVACITY_ONLY_SENSOR = ""    # e.g. "2158" to fetch one sensor only
VIVACITY_MERGE_EXISTING = True

REFRESH_VIVACITY_MANIFEST = True

ECO_COUNTER_API = os.environ.get("ECO_COUNTER_API")
VIVACITY_API = os.environ.get("VIVACITY_API")

if not ECO_COUNTER_API:
    raise ValueError("Set ECO_COUNTER_API in the environment or .env")
if not VIVACITY_API:
    print("Warning: VIVACITY_API not set — Vivacity section will be skipped.")

CWD = Path.cwd().resolve()
REPO_ROOT = CWD if (CWD / "static").is_dir() else CWD.parent
STATIC = REPO_ROOT / "static"
DATA = STATIC / "data"
DATA.mkdir(parents=True, exist_ok=True)

ECO_OUT = DATA / "eco-counter-weekly-snapshot.json"
VIVACITY_OUT = DATA / "vivacity-weekly-snapshot.json"
VIVACITY_MANIFEST = DATA / "vivacity-sensor-manifest.json"

print(f"Repo root: {REPO_ROOT}")
print(f"Eco output: {ECO_OUT.relative_to(REPO_ROOT)}")
print(f"Vivacity output: {VIVACITY_OUT.relative_to(REPO_ROOT)}")


Repo root: /Users/rudi/Downloads/atd-v3
Eco output: static/data/eco-counter-weekly-snapshot.json
Vivacity output: static/data/vivacity-weekly-snapshot.json


In [4]:
def utc_now_iso():
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def monday_week_key(dt):
    """ISO week start (Monday) as YYYY-MM-DD in UTC."""
    if isinstance(dt, str):
        dt = datetime.fromisoformat(dt.replace("Z", "+00:00"))
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    monday = dt - timedelta(days=dt.weekday())
    return monday.date().isoformat()


def fmt_eco_date(dt):
    return dt.strftime("%Y-%m-%d")


def fmt_vivacity_utc(dt):
    return dt.astimezone(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def snapshot_history_start(end=None):
    """First day of the YoY comparison month (UTC).

    e.g. run on 8 Jul 2026 → last completed month is Jun 2026 → start 1 Jun 2025.
    """
    end = end or datetime.now(timezone.utc)
    if end.tzinfo is None:
        end = end.replace(tzinfo=timezone.utc)
    first_of_current = end.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    last_completed = first_of_current - timedelta(days=1)
    last_completed_start = last_completed.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    return last_completed_start.replace(year=last_completed_start.year - 1)


def snapshot_history_weeks(start, end):
    days = max(1, (end.date() - start.date()).days)
    return (days + 6) // 7


def snapshot_yoy_month_keys(end=None):
    """(prior_year_month, last_completed_month) as YYYY-MM."""
    start = snapshot_history_start(end)
    end = end or datetime.now(timezone.utc)
    if end.tzinfo is None:
        end = end.replace(tzinfo=timezone.utc)
    first_of_current = end.replace(day=1, hour=0, minute=0, second=0, microsecond=0)
    last_completed = first_of_current - timedelta(days=1)
    last_completed_start = last_completed.replace(day=1)
    return start.strftime("%Y-%m"), last_completed_start.strftime("%Y-%m")


def write_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
        f.write("\n")
    print(f"Wrote {path.relative_to(REPO_ROOT)} ({path.stat().st_size // 1024} KB)")


def rollup_daily_to_weekly(daily_rows):
    """daily_rows: list of {dateKey, pedestrian, bike} -> weekly totals."""
    buckets = defaultdict(lambda: {"pedestrian": 0, "bike": 0})
    for row in daily_rows:
        wk = monday_week_key(row["dateKey"])
        buckets[wk]["pedestrian"] += int(row.get("pedestrian") or 0)
        buckets[wk]["bike"] += int(row.get("bike") or 0)
    return [
        {"weekKey": wk, **buckets[wk]}
        for wk in sorted(buckets)
    ]


## Eco-Counter — weekly per site

Fetches `granularity=P1W` from the Eco-Counter API (one row per week per travel mode per site). Requests are serialised with pauses and 429 backoff.

In [5]:
ECO_HEADERS = {"accept": "application/json", "X-API-KEY": ECO_COUNTER_API}


def normalize_eco_travel_mode(raw):
    if raw is None:
        return None
    s = str(raw).lower()
    if s in ("pedestrian", "walker", "walking"):
        return "pedestrian"
    if s in ("bike", "bicycle", "cyclist"):
        return "bike"
    return None


def extract_eco_travel_mode_series(payload):
    out = []

    def push_series(tm_raw, data):
        mode = normalize_eco_travel_mode(tm_raw)
        if not mode or not isinstance(data, list) or not data:
            return
        out.append({"travelMode": mode, "data": data})

    def consume_flow_array(flows):
        if not isinstance(flows, list):
            return
        for flow in flows:
            if not isinstance(flow, dict):
                continue
            tm = flow.get("travelMode") or flow.get("travel_mode") or flow.get("mode") or flow.get("userType")
            data = flow.get("data") or flow.get("points") or flow.get("values") or flow.get("records") or flow.get("items") or flow.get("intervals")
            push_series(tm, data)

    if not payload:
        return out

    if isinstance(payload, list):
        if payload and isinstance(payload[0], dict) and isinstance(payload[0].get("data"), list):
            consume_flow_array(payload)
            if out:
                return out
        by_mode = defaultdict(list)
        for row in payload:
            if not isinstance(row, dict):
                continue
            tm = normalize_eco_travel_mode(row.get("travelMode") or row.get("travel_mode") or row.get("mode") or row.get("userType"))
            if tm:
                by_mode[tm].append(row)
        for tm, rows in by_mode.items():
            out.append({"travelMode": tm, "data": rows})
        return out

    if isinstance(payload, dict):
        for ped_key in ("pedestrian", "walker", "walking"):
            if isinstance(payload.get(ped_key), list) and payload[ped_key]:
                push_series(ped_key, payload[ped_key])
                break
        for bike_key in ("bike", "bicycle", "cyclist"):
            if isinstance(payload.get(bike_key), list) and payload[bike_key]:
                push_series(bike_key, payload[bike_key])
                break
        if out:
            return out
        for key in ("data", "content", "items", "flows", "series", "traffic"):
            block = payload.get(key)
            if isinstance(block, list):
                consume_flow_array(block)
                if out:
                    return out
        consume_flow_array(payload.get("flows"))
    return out


def eco_point_count(point):
    for k in ("counts", "count", "value", "total", "volume"):
        if point.get(k) is not None:
            try:
                return int(float(point[k]))
            except (TypeError, ValueError):
                pass
    traffic = point.get("traffic")
    if isinstance(traffic, dict):
        for k in ("counts", "value"):
            if traffic.get(k) is not None:
                try:
                    return int(float(traffic[k]))
                except (TypeError, ValueError):
                    pass
    return 0


def eco_point_date(point):
    for k in ("timestamp", "isoDate", "iso_date", "from", "startDate", "start", "date", "period"):
        if point.get(k):
            return str(point[k])
    return None


def weekly_rows_from_eco_payload(payload):
    series_list = extract_eco_travel_mode_series(payload)
    buckets = defaultdict(lambda: {"pedestrian": 0, "bike": 0})
    for series in series_list:
        mode = series["travelMode"]
        for point in series["data"]:
            date_str = eco_point_date(point)
            if not date_str:
                continue
            try:
                dt = datetime.fromisoformat(date_str.replace("Z", "+00:00"))
            except ValueError:
                continue
            wk = monday_week_key(dt)
            buckets[wk][mode] += eco_point_count(point)
    return [{"weekKey": wk, **buckets[wk]} for wk in sorted(buckets)]


def eco_get(url):
    backoff = ECO_429_BACKOFF_S
    for attempt in range(ECO_429_MAX_RETRIES + 1):
        r = requests.get(url, headers=ECO_HEADERS, timeout=120)
        if r.status_code == 429:
            if attempt >= ECO_429_MAX_RETRIES:
                r.raise_for_status()
            wait = backoff
            print(f"  Eco 429 — waiting {wait}s before retry {attempt + 1}/{ECO_429_MAX_RETRIES}")
            time.sleep(wait)
            backoff *= 2
            continue
        r.raise_for_status()
        if ECO_PAUSE_S:
            time.sleep(ECO_PAUSE_S)
        return r.json()
    raise RuntimeError("eco_get: unreachable")


def eco_sites_list(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ("data", "content", "items"):
            if isinstance(payload.get(key), list):
                return payload[key]
    return []


end = datetime.now(timezone.utc)
start = snapshot_history_start(end)
dd_start, dd_end = fmt_eco_date(start), fmt_eco_date(end)
history_weeks = snapshot_history_weeks(start, end)
yoy_prior_month, yoy_current_month = snapshot_yoy_month_keys(end)

sites_json = eco_get("https://api.eco-counter.com/api/v2/sites?page=1&pageSize=100&sortBy=id&orderBy=asc")
traffic_url = (
    "https://api.eco-counter.com/api/v2/statistical/adt/by/site"
    "?dateRange=lastMonth&groupBy=siteAndTravelMode&travelModes=pedestrian&travelModes=bike"
)
traffic_json = None
try:
    traffic_json = eco_get(traffic_url)
except requests.HTTPError as e:
    print("ADT fetch failed:", e)

site_meta = {}
for site in eco_sites_list(sites_json):
    sid = site.get("id") or site.get("siteId")
    if sid is None:
        continue
    site_meta[int(sid)] = {
        "name": site.get("name") or f"Site {sid}",
        "travelModes": site.get("travelModes") or [],
    }

if isinstance(traffic_json, list):
    for row in traffic_json:
        sid = row.get("siteId")
        if sid is None:
            continue
        tm = normalize_eco_travel_mode(row.get("travelMode"))
        if tm:
            site_meta.setdefault(int(sid), {"name": f"Site {sid}", "travelModes": []})
            if tm not in site_meta[int(sid)]["travelModes"]:
                site_meta[int(sid)]["travelModes"].append(tm)

site_ids = sorted(site_meta.keys())
print(f"Eco sites: {len(site_ids)}, window {dd_start} → {dd_end} ({history_weeks} weeks)")
print(f"YoY months: {yoy_prior_month} vs {yoy_current_month}")


def fetch_eco_site_weekly(site_id):
    url = (
        "https://api.eco-counter.com/api/v2/history/traffic/aggregated?"
        + urlencode({
            "siteId": site_id,
            "include": "",
            "startDate": dd_start,
            "endDate": dd_end,
            "startTime": "00:00",
            "endTime": "00:00",
            "granularity": "P1W",
            "groupBy": "travelMode",
            "gapFilling": "false",
            "travelModes": ["pedestrian", "bike"],
        }, doseq=True)
    )
    try:
        payload = eco_get(url)
        return site_id, weekly_rows_from_eco_payload(payload), None
    except Exception as e:
        return site_id, [], str(e)


eco_sites_out = {}
errors = []
done = 0
with ThreadPoolExecutor(max_workers=ECO_CONCURRENCY) as pool:
    futures = {pool.submit(fetch_eco_site_weekly, sid): sid for sid in site_ids}
    for fut in as_completed(futures):
        site_id, weekly, err = fut.result()
        done += 1
        if err:
            errors.append((site_id, err))
        meta = site_meta.get(site_id, {})
        eco_sites_out[str(site_id)] = {
            "siteId": site_id,
            "name": meta.get("name", f"Site {site_id}"),
            "travelModes": meta.get("travelModes", []),
            "weekly": weekly,
        }
        if done % 5 == 0 or done == len(site_ids):
            print(f"Eco progress {done}/{len(site_ids)}")

eco_payload = {
    "schemaVersion": 1,
    "granularity": "P1W",
    "generatedAtUtc": utc_now_iso(),
    "historyWeeks": history_weeks,
    "yoyCompareMonths": {"prior": yoy_prior_month, "current": yoy_current_month},
    "dateRange": {"startDate": dd_start, "endDate": dd_end},
    "siteCount": len(eco_sites_out),
    "sites": eco_sites_out,
}
write_json(ECO_OUT, eco_payload)
if errors:
    print(f"Eco errors ({len(errors)}):", errors[:5])


Eco sites: 88, window 2025-06-01 → 2026-07-08 (58 weeks)
YoY months: 2025-06 vs 2026-06
Eco progress 5/88
Eco progress 10/88
Eco progress 15/88
Eco progress 20/88
Eco progress 25/88
Eco progress 30/88
Eco progress 35/88
Eco progress 40/88
Eco progress 45/88
Eco progress 50/88
Eco progress 55/88
Eco progress 60/88
Eco progress 65/88
Eco progress 70/88
Eco progress 75/88
Eco progress 80/88
Eco progress 85/88
Eco progress 88/88
Wrote static/data/eco-counter-weekly-snapshot.json (319 KB)


## Vivacity — weekly per sensor

Fetches `time_bucket=24h` per countline in short windows, rolls up to **weekly totals in memory**, and writes only weekly rows.

In [6]:
import subprocess

VIVACITY_COUNTS_BASE = "https://api.vivacitylabs.com/countline/counts"
VIVACITY_COUNTS_CLASSES = "classes=pedestrian&classes=cyclist"


def vivacity_headers():
    return {"Accept": "application/json", "x-vivacity-api-key": VIVACITY_API}


def with_vivacity_classes(url):
    return url if "classes=pedestrian" in url else f"{url}&{VIVACITY_COUNTS_CLASSES}"


def utc_day_start(dt):
    dt = dt.astimezone(timezone.utc)
    return datetime(dt.year, dt.month, dt.day, tzinfo=timezone.utc)


def iter_time_windows(from_day, to_day, max_days):
    out = []
    cur = from_day
    to_day = utc_day_start(to_day)
    while cur < to_day:
        nxt = min(cur + timedelta(days=max_days), to_day)
        out.append((cur, nxt))
        cur = nxt
    return out


def aggregate_vivacity_data(data):
    if not isinstance(data, dict):
        return []
    keys = [k for k in data if isinstance(data[k], list)]
    if not keys:
        return []
    first_series = data[keys[0]]
    aggregated = []
    for time_entry in first_series:
        entry = {"from": time_entry.get("from"), "to": time_entry.get("to"), "pedestrian": 0, "cyclist": 0}
        for clid in keys:
            rows = data[clid]
            if not isinstance(rows, list):
                continue
            matching = next((e for e in rows if isinstance(e, dict) and e.get("from") == time_entry.get("from")), None)
            if not matching:
                continue
            for direction in ("clockwise", "anti_clockwise"):
                d = matching.get(direction)
                if not isinstance(d, dict):
                    continue
                for vehicle_type, count in d.items():
                    if vehicle_type in entry:
                        entry[vehicle_type] += int(count or 0)
        aggregated.append(entry)
    return aggregated


def merge_rows_by_from(a, b):
    m = {}
    for r in a + b:
        if isinstance(r, dict) and r.get("from"):
            m[str(r["from"])] = r
    return sorted(m.values(), key=lambda x: str(x.get("from", "")))


def vivacity_daily_rows_to_weekly(aggregated_rows):
    daily = []
    for row in aggregated_rows:
        from_iso = row.get("from")
        if not from_iso:
            continue
        daily.append({
            "dateKey": from_iso,
            "pedestrian": int(row.get("pedestrian") or 0),
            "bike": int(row.get("cyclist") or 0),
        })
    return rollup_daily_to_weekly(daily)


def fetch_countline_daily_rows(countline_id, from_day, to_day):
    merged = []
    windows = iter_time_windows(from_day, to_day, VIVACITY_WINDOW_MAX_DAYS)
    for w_from, w_to in windows:
        url = with_vivacity_classes(
            f"{VIVACITY_COUNTS_BASE}?countline_ids={countline_id}"
            f"&from={fmt_vivacity_utc(w_from)}&to={fmt_vivacity_utc(w_to)}&time_bucket=24h"
        )
        r = requests.get(url, headers=vivacity_headers(), timeout=VIVACITY_TIMEOUT_S)
        if r.status_code == 204 or not r.text.strip():
            rows = []
        else:
            r.raise_for_status()
            part = r.json()
            rows = part.get(str(countline_id)) or part.get(countline_id) or []
            if not isinstance(rows, list):
                rows = []
        merged = merge_rows_by_from(merged, rows)
        if VIVACITY_PAUSE_S:
            time.sleep(VIVACITY_PAUSE_S)
    return merged


def fetch_sensor_weekly(sensor_id, countline_ids, from_day, to_day):
    merged_raw = {}
    for clid in countline_ids:
        merged_raw[clid] = fetch_countline_daily_rows(clid, from_day, to_day)
    aggregated = aggregate_vivacity_data(merged_raw)
    return vivacity_daily_rows_to_weekly(aggregated)


if not VIVACITY_API:
    print("Skipping Vivacity — no API key.")
elif REFRESH_VIVACITY_MANIFEST:
    print("Refreshing vivacity-sensor-manifest.json …")
    subprocess.run(["npm", "run", "export:vivacity-manifest"], cwd=REPO_ROOT, check=True)

if VIVACITY_API and VIVACITY_MANIFEST.exists():
    manifest = json.loads(VIVACITY_MANIFEST.read_text())
    sensors = manifest.get("sensors") or []
    if VIVACITY_ONLY_SENSOR:
        sensors = [s for s in sensors if str(s.get("sensor_id")) == VIVACITY_ONLY_SENSOR]

    v_end = utc_day_start(datetime.now(timezone.utc))
    v_start = utc_day_start(snapshot_history_start(v_end))
    history_weeks = snapshot_history_weeks(v_start, v_end)
    yoy_prior_month, yoy_current_month = snapshot_yoy_month_keys(v_end)

    existing = {}
    if VIVACITY_MERGE_EXISTING and VIVACITY_OUT.exists():
        try:
            existing = json.loads(VIVACITY_OUT.read_text()).get("sensors") or {}
        except json.JSONDecodeError:
            existing = {}

    vivacity_sensors_out = dict(existing) if VIVACITY_MERGE_EXISTING and VIVACITY_ONLY_SENSOR else {}
    print(f"Vivacity sensors to fetch: {len(sensors)}, window {fmt_vivacity_utc(v_start)} → {fmt_vivacity_utc(v_end)} ({history_weeks} weeks)")
    print(f"YoY months: {yoy_prior_month} vs {yoy_current_month}")

    for i, sensor in enumerate(sensors, 1):
        sid = str(sensor.get("sensor_id"))
        clids = [str(c) for c in (sensor.get("countline_ids") or [])]
        if not sid or not clids:
            continue
        print(f"[{i}/{len(sensors)}] sensor {sid} ({len(clids)} countlines)")
        try:
            weekly = fetch_sensor_weekly(sid, clids, v_start, v_end)
            vivacity_sensors_out[sid] = {
                "sensorId": sid,
                "countlineIds": clids,
                "weekly": weekly,
            }
        except Exception as e:
            print(f"  FAILED sensor {sid}: {e}")

    vivacity_payload = {
        "schemaVersion": 1,
        "granularity": "P1W",
        "generatedAtUtc": utc_now_iso(),
        "historyWeeks": history_weeks,
        "yoyCompareMonths": {"prior": yoy_prior_month, "current": yoy_current_month},
        "dateRange": {"from": fmt_vivacity_utc(v_start), "to": fmt_vivacity_utc(v_end)},
        "sensorCount": len(vivacity_sensors_out),
        "sensors": vivacity_sensors_out,
    }
    write_json(VIVACITY_OUT, vivacity_payload)
elif VIVACITY_API:
    print(f"Missing manifest at {VIVACITY_MANIFEST} — run with REFRESH_VIVACITY_MANIFEST=True")


Refreshing vivacity-sensor-manifest.json …

> my-app@0.0.1 export:vivacity-manifest
> node scripts/export-vivacity-sensor-manifest.mjs

Wrote /Users/rudi/Downloads/atd-v3/static/data/vivacity-sensor-manifest.json { sensorCount: 22 }
Vivacity sensors to fetch: 22, window 2025-06-01T00:00:00Z → 2026-07-08T00:00:00Z (58 weeks)
YoY months: 2025-06 vs 2026-06
[1/22] sensor 2158 (3 countlines)
[2/22] sensor 2159 (3 countlines)
[3/22] sensor 3763 (3 countlines)
[4/22] sensor 3771 (4 countlines)
[5/22] sensor 4158 (3 countlines)
[6/22] sensor 4159 (4 countlines)
[7/22] sensor 7487 (5 countlines)
[8/22] sensor 8479 (2 countlines)
[9/22] sensor 9510 (6 countlines)
[10/22] sensor 9646 (2 countlines)
[11/22] sensor 9658 (4 countlines)
[12/22] sensor 9712 (9 countlines)
[13/22] sensor 9713 (6 countlines)
[14/22] sensor 9714 (3 countlines)
[15/22] sensor 9716 (5 countlines)
[16/22] sensor 9719 (3 countlines)
[17/22] sensor 9739 (2 countlines)
[18/22] sensor 9740 (2 countlines)
[19/22] sensor 9741 (2